In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, '..')
import config
from src.data_loader import load_data
from src.pair_selector import run_pair_selection, get_selected_pairs
from src.signal_generator import (
    generate_signals, generate_signals_for_pair,
    compute_rolling_beta_fast, compute_rolling_spread, compute_rolling_zscore
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Load data and selected pairs
prices = load_data(force_refresh=False)

# Load saved pairs from pair selection, otherwise re-run
try:
    selected = pd.read_csv('../data/selected_pairs.csv')
    print(f"✅ Loaded {len(selected)} selected pairs from Phase 2")
    for _, row in selected.iterrows():
        a = row['stock_a'].replace('.NS', '')
        b = row['stock_b'].replace('.NS', '')
        print(f"   {a} ↔ {b} ({row['sector']})")
except FileNotFoundError:
    print("⚠️  No saved pairs found. Running pair selection...")
    results = run_pair_selection(prices, verbose=True)
    selected = get_selected_pairs(results)
    selected.to_csv('./data/selected_pairs.csv', index=False)

## 1. Generate Signals for All Selected Pairs

For each pair, we compute:
- **Rolling β** — hedge ratio that adapts over time (60-day window)
- **Rolling spread** — log(A) - β × log(B)
- **Rolling z-score** — how many std devs the spread is from its rolling mean
- **Signal** — +1 (long), -1 (short), 0 (flat)
- **Risk overlay** — stop-loss and max holding period

All computed on the **test period only** (last 1 year of data).

In [ ]:
# Generate signals on the test period
all_signals = generate_signals(prices, selected, use_test_period=True, verbose=True)

## 2. Visualize: Price, Spread, Z-Score & Signals

For each pair, we create a 3-panel chart:
- **Top:** Both stock prices with trade markers
- **Middle:** Spread with rolling mean
- **Bottom:** Z-score with entry/exit thresholds and signal shading

In [ ]:
def plot_pair_signals(pair_name, signals_df):
    """Create a detailed 3-panel chart for one pair."""
    
    a_name, b_name = pair_name.split('-')
    
    fig = make_subplots(
        rows=3, cols=1, row_heights=[0.33, 0.33, 0.34],
        shared_xaxes=True,
        subplot_titles=[
            f'Stock Prices: {a_name} vs {b_name}',
            'Spread (log(A) - β × log(B))',
            'Z-Score & Signals'
        ],
        vertical_spacing=0.06
    )
    
    df = signals_df.copy()
    
    # --- Panel 1: Stock Prices ---
    # Normalize both to 100 for comparison
    norm_a = df['price_a'] / df['price_a'].iloc[0] * 100
    norm_b = df['price_b'] / df['price_b'].iloc[0] * 100
    
    fig.add_trace(go.Scatter(
        x=df.index, y=norm_a, name=a_name,
        line=dict(color='#00d4aa', width=1.5)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=norm_b, name=b_name,
        line=dict(color='#ff6b6b', width=1.5)
    ), row=1, col=1)
    
    # Add trade entry markers on price chart
    entries = df[df['signal'].diff().ne(0) & (df['signal'] != 0)]
    for idx, row in entries.iterrows():
        color = '#00ff88' if row['signal'] == 1 else '#ff4444'
        symbol = 'triangle-up' if row['signal'] == 1 else 'triangle-down'
        norm_val = norm_a.loc[idx]
        fig.add_trace(go.Scatter(
            x=[idx], y=[norm_val], mode='markers',
            marker=dict(size=10, color=color, symbol=symbol),
            showlegend=False
        ), row=1, col=1)
    
    # --- Panel 2: Spread ---
    fig.add_trace(go.Scatter(
        x=df.index, y=df['spread'], name='Spread',
        line=dict(color='#6b9fff', width=1)
    ), row=2, col=1)
    
    # Rolling mean of spread
    spread_mean = df['spread'].rolling(config.LOOKBACK_WINDOW).mean()
    fig.add_trace(go.Scatter(
        x=df.index, y=spread_mean, name='Rolling Mean',
        line=dict(color='yellow', width=1, dash='dash')
    ), row=2, col=1)
    
    # --- Panel 3: Z-Score with signal shading ---
    fig.add_trace(go.Scatter(
        x=df.index, y=df['zscore'], name='Z-Score',
        line=dict(color='#b388ff', width=1.5)
    ), row=3, col=1)
    
    # Threshold lines
    fig.add_hline(y=config.Z_ENTRY_THRESHOLD, row=3, col=1,
                  line_dash='dash', line_color='red', annotation_text=f'+{config.Z_ENTRY_THRESHOLD}')
    fig.add_hline(y=-config.Z_ENTRY_THRESHOLD, row=3, col=1,
                  line_dash='dash', line_color='red', annotation_text=f'-{config.Z_ENTRY_THRESHOLD}')
    fig.add_hline(y=config.Z_EXIT_THRESHOLD, row=3, col=1,
                  line_dash='dot', line_color='green', opacity=0.5)
    fig.add_hline(y=-config.Z_EXIT_THRESHOLD, row=3, col=1,
                  line_dash='dot', line_color='green', opacity=0.5)
    fig.add_hline(y=0, row=3, col=1, line_color='gray', opacity=0.3)
    
    # Shade long/short signal regions
    long_periods = df[df['signal'] == 1]
    short_periods = df[df['signal'] == -1]
    
    if len(long_periods) > 0:
        fig.add_trace(go.Scatter(
            x=long_periods.index, y=long_periods['zscore'],
            mode='markers', marker=dict(color='rgba(0,255,136,0.3)', size=4),
            name='Long Signal', showlegend=True
        ), row=3, col=1)
    
    if len(short_periods) > 0:
        fig.add_trace(go.Scatter(
            x=short_periods.index, y=short_periods['zscore'],
            mode='markers', marker=dict(color='rgba(255,68,68,0.3)', size=4),
            name='Short Signal', showlegend=True
        ), row=3, col=1)
    
    # Layout
    fig.update_layout(
        height=800, template='plotly_dark',
        title=f'{pair_name} — Signal Analysis (Test Period)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig.update_yaxes(title_text='Normalized Price', row=1, col=1)
    fig.update_yaxes(title_text='Spread', row=2, col=1)
    fig.update_yaxes(title_text='Z-Score', row=3, col=1)
    
    return fig

# Plot all pairs
for pair_name, signals_df in all_signals.items():
    fig = plot_pair_signals(pair_name, signals_df)
    fig.show()

## 3. Signal Statistics Summary

For each pair, let's see:
- How many trades were generated
- What % of time we're in a position vs flat
- Average z-score at entry

In [ ]:
summary = []

for pair_name, df in all_signals.items():
    sig = df['signal']
    z = df['zscore']
    
    # Count trades (each transition from 0 to ±1 is an entry)
    signal_changes = sig.diff().ne(0)
    entries = ((sig != 0) & signal_changes)
    n_trades = entries.sum()
    
    # Time in position
    long_pct = (sig == 1).mean() * 100
    short_pct = (sig == -1).mean() * 100
    flat_pct = (sig == 0).mean() * 100
    
    # Average z-score at entry
    entry_zscores = z[entries].dropna()
    avg_entry_z = entry_zscores.abs().mean() if len(entry_zscores) > 0 else 0
    
    # Average holding period
    trade_lengths = []
    current_length = 0
    for s in sig:
        if s != 0:
            current_length += 1
        elif current_length > 0:
            trade_lengths.append(current_length)
            current_length = 0
    if current_length > 0:
        trade_lengths.append(current_length)
    avg_hold = np.mean(trade_lengths) if trade_lengths else 0
    
    summary.append({
        'Pair': pair_name,
        'Trades': int(n_trades),
        'Long %': round(long_pct, 1),
        'Short %': round(short_pct, 1),
        'Flat %': round(flat_pct, 1),
        'Avg |z| Entry': round(avg_entry_z, 2),
        'Avg Hold (days)': round(avg_hold, 1),
    })

summary_df = pd.DataFrame(summary)
print("\n📊 Signal Statistics:")
summary_df

## 4. Rolling Beta Stability

The hedge ratio β should be relatively stable over time.
If it's jumping wildly, the relationship between stocks is unstable.

In [ ]:
# Plot rolling beta for all pairs
n_pairs = len(all_signals)
fig = make_subplots(rows=n_pairs, cols=1,
                    subplot_titles=list(all_signals.keys()),
                    vertical_spacing=0.04)

for i, (pair_name, df) in enumerate(all_signals.items(), 1):
    fig.add_trace(go.Scatter(
        x=df.index, y=df['beta'], name=pair_name,
        line=dict(width=1.5), showlegend=False
    ), row=i, col=1)
    fig.update_yaxes(title_text='β', row=i, col=1)

fig.update_layout(
    height=200 * n_pairs, template='plotly_dark',
    title='Rolling Hedge Ratio (β) — Should be relatively stable'
)
fig.show()

## 5. Risk Overlay Impact

Compare raw signals vs risk-adjusted signals.
How many trades were cut short by stop-loss or max holding rules?

In [ ]:
risk_impact = []

for pair_name, df in all_signals.items():
    raw_in_position = (df['raw_signal'] != 0).sum()
    adj_in_position = (df['signal'] != 0).sum()
    
    raw_entries = ((df['raw_signal'] != 0) & df['raw_signal'].diff().ne(0)).sum()
    adj_entries = ((df['signal'] != 0) & df['signal'].diff().ne(0)).sum()
    
    days_cut = raw_in_position - adj_in_position
    
    risk_impact.append({
        'Pair': pair_name,
        'Raw Entries': int(raw_entries),
        'Adj Entries': int(adj_entries),
        'Raw Days In': int(raw_in_position),
        'Adj Days In': int(adj_in_position),
        'Days Cut': int(days_cut),
    })

risk_df = pd.DataFrame(risk_impact)
print("\n🛡️ Risk Overlay Impact:")
risk_df

In [ ]:
import os

# Save each pair's signals
signals_dir = '../data/signals'
os.makedirs(signals_dir, exist_ok=True)

for pair_name, df in all_signals.items():
    filepath = os.path.join(signals_dir, f'{pair_name}.parquet')
    df.to_parquet(filepath)

print(f"✅ Saved signals for {len(all_signals)} pairs to {signals_dir}/")